In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [ ]:
# LSTM : 시간 순서가 있는 데이터를 처리하는 레이어
# 일반 신경망은 이전 입력을 기억 못하는데, LSTM은 기억

# 레이어 설계
# 단어는 보통 6차원 벡터로 표시되기 때문에 EMBEDDING_DIM을 6으로 할당하고
# 공정 같은 경우는 시점 마다 전류, RPM이 표시하는 숫자값이 배열 안에 각각의 형태로 들어올 것이기 때문에 특징 수가 2개이다.
lstm = nn.LSTM(3, 3)
#              ↑  ↑
#          특징수  기억크기

# 예시 데이터
inputs = [torch.randn(1, 1, 3) for _ in range(5)]
#                     ↑  ↑  ↑
#                   seq 배치 특징수
# 예를 들면
# 3번째 매개변수는 특징수이기 때문에 [전류, RPM, 온도]  이렇게 3개가 예시가 될 수 있다.

hidden = (torch.rand(1, 1, 6),  # h_n : 단기 기억
          torch.rand(1, 1, 6))  # c_n : 장기 기억
#                    ↑  ↑  ↑
#   (형태 맞추려고) seq 배치 기억크기

for i in inputs:
  # 레이어 실행
  out, hidden = lstm(i.view(1, 1, -1), hidden)
  #                  ↑                 ↑
  #                3D 입력형식          이전 기억
  # 3D로 만드는 이유는 lstm이 받을 수 있는 형태이기 때문이다.

# 위의 과정과 같지만 코드 줄을 줄이는 방법 (for문을 사용하지 않음)
inputs = torch.cat(inputs).view(len(inputs), 1, -1)
hidden = (torch.rand(1, 1, 3), torch.rand(1, 1, 3))

out, hidden = lstm(inputs, hidden)

In [35]:
# 단어를 숫자로 바꾸는 함수
# sequence : 순서 있는 데이터(LSTM의 핵심)
def prepare_sequence(seq, to_ix):
  idxs = [to_ix[w] for w in seq]
  return torch.tensor(idxs, dtype=torch.long)

training_data = [
  # 첫 번째 : 순서 있는 입력 데이터
  # 두 번째 : 각 입력에 대한 정답 레이블
  ("The dog ate the apple.".split(), ["DET", "NN", "V", "DET", "NN"]),
  ("Everybody read that book".split(), ["NN", "V", "DET", "NN"])
]

# 단어를 index화
word_to_ix = {}

# 중복되는 단어를 제외하고 단어의 순서를 index화
for sent, tags in training_data:
  for word in sent:
    if word not in word_to_ix:
      word_to_ix[word] = len(word_to_ix)


print(word_to_ix)

# 품사를 숫자로 바꾸는 딕셔너리
tag_to_ix = {"DET": 0, "NN": 1, "V": 2}

# 단어를 몇 차원 벡터로 표현할지 크기
EMBEDDING_DIM = 6
# 기억 크기
HIDDEN_DIM = 6

{'The': 0, 'dog': 1, 'ate': 2, 'the': 3, 'apple.': 4, 'Everybody': 5, 'read': 6, 'that': 7, 'book': 8}


In [36]:
class LSTMTagger(nn.Module):
  def __init__(self, embedding_dim, hidden_dim, vocab_size, target_size):
    super().__init__()
    self.hidden_dim = hidden_dim

    # Embedding : 단어를 벡터로 바꾸는 것
    self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

    # embedding_dim
    # 레이어 설계
    self.lstm = nn.LSTM(embedding_dim, hidden_dim)

    # 기억 크기를 품사의 정답에 맞게 압축
    # 기억 크기를 target_size(정답)의 크기 만큼 줄이기
    self.hidden2tag = nn.Linear(hidden_dim, target_size)

  def forward(self, sentence):
    embeds = self.word_embeddings(sentence)
    lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
    tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
    tag_scores = F.log_softmax(tag_space, dim=1)
    return tag_scores

In [37]:
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

with torch.no_grad():
  inputs = prepare_sequence(training_data[0][0], word_to_ix)
  tag_scores = model(inputs)

for epoch in range(300):
  for sentence, tags in training_data:

    model.zero_grad()

    sentence_in = prepare_sequence(sentence, word_to_ix)
    targets = prepare_sequence(tags, tag_to_ix)

    tag_scores = model(sentence_in)

    loss = loss_function(tag_scores, targets)
    loss.backward()
    optimizer.step()

with torch.no_grad():
  inputs = prepare_sequence(training_data[0][0], word_to_ix)
  tag_scores = model(inputs)

  print(tag_scores)

tensor([[-0.0471, -3.8399, -3.7075],
        [-4.2119, -0.0466, -3.4840],
        [-2.8518, -3.9280, -0.0806],
        [-0.0720, -3.2803, -3.4470],
        [-3.9195, -0.0266, -5.0474]])
